# BioFreshGuard

## Ecological Freshness Intelligence System

BioFreshGuard is a data-driven ecological system for analyzing post-harvest freshness and spoilage dynamics of lettuce under different biological treatments.

The system uses only the original experimental criteria from the database:

- morphology
- bacteria
- weight
- color
- Texture analyzer
- TSS
- O2
- CO2

The goal is to build a complete ecological analysis pipeline that includes:
1. Ecological system definition based on Odum model
2. Research hypotheses and variables
3. PCA analysis
4. Statistical modeling
5. Machine learning prediction
6. Spatial simulation
7. Interactive dashboard

In [55]:
!pip install pandas numpy matplotlib scikit-learn scipy statsmodels plotly ipywidgets openpyxl -q

In [56]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from functools import reduce

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

import plotly.express as px
import plotly.graph_objects as go

from google.colab import files

import warnings
warnings.filterwarnings("ignore")

In [57]:
uploaded = files.upload()

Saving BFG_DATABASE.xlsx to BFG_DATABASE (1).xlsx


In [58]:
file_name = list(uploaded.keys())[0]

xls = pd.ExcelFile(file_name)
sheet_names = xls.sheet_names

print("Sheets found in BFG database:")
for sheet in sheet_names:
    print("-", sheet)

Sheets found in BFG database:
- morphology
- bacteria
- weight
- color
- Texture analyzer
- TSS
- O2
- CO2


In [59]:
expected_sheets = [
    "morphology",
    "bacteria",
    "weight",
    "color",
    "Texture analyzer",
    "TSS",
    "O2",
    "CO2"
]

missing_sheets = [s for s in expected_sheets if s not in sheet_names]

if len(missing_sheets) == 0:
    print("Database validation passed: all expected sheets exist.")
else:
    print("Warning: missing sheets:", missing_sheets)

Database validation passed: all expected sheets exist.


In [60]:
raw_sheets = {}

for sheet in expected_sheets:
    df = pd.read_excel(file_name, sheet_name=sheet)
    raw_sheets[sheet] = df

    print("\n" + "="*60)
    print("Sheet:", sheet)
    print("Shape:", df.shape)
    display(df.head())


Sheet: morphology
Shape: (20, 3)


,Time (day),Treatment,morphology_rate
0,0,control,5.00000
1,4,control,5.00000
2,7,control,4.00000
3,14,control,2.66667
4,0,CMC-3-methyl,5.00000



Sheet: bacteria
Shape: (20, 5)


,Time (day),Treatment,bacteria_total,yeasts,lactic
0,0,control,6000.00,0,0.0
1,4,control,13888.89,0,220000.0
2,7,control,19805.56,0,385000.0
3,14,control,33611.11,0,770000.0
4,0,CMC-3-methyl,6555.56,0,0.0



Sheet: weight
Shape: (20, 3)


...

,Time (day),Treatment,TSS (%)
0,0,control,5.46667
1,4,control,6.10000
2,7,control,5.60000
3,14,control,4.43333
4,0,CMC-3-methyl,4.33333



Sheet: O2
Shape: (20, 3)


,Time (day),Treatment,%O2
0,0,control,21.00000
1,4,control,20.61234
2,7,control,13.62778
3,14,control,0.00000
4,0,CMC-3-methyl,21.00000



Sheet: CO2
Shape: (20, 3)


,Time (day),Treatment,co2
0,0,control,0.00000
1,4,control,0.34225
2,7,control,1.11594
3,14,control,2.92122
4,0,CMC-3-methyl,0.00000


In [61]:
def clean_column_name(col):
    return (
        str(col)
        .strip()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("%", "percent")
        .replace("-", "_")
        .replace("/", "_")
    )

clean_sheets = {}

for sheet, df in raw_sheets.items():
    df = df.copy()
    df.columns = [clean_column_name(c) for c in df.columns]
    clean_sheets[sheet] = df

for sheet, df in clean_sheets.items():
    print(sheet, "=>", list(df.columns))

morphology => ['Time_day', 'Treatment', 'morphology_rate']
bacteria => ['Time_day', 'Treatment', 'bacteria_total', 'yeasts', 'lactic']
weight => ['Time_day', 'Treatment', 'weight_loss_percent']
color => ['Time_day', 'Treatment', 'delta_E']
Texture analyzer => ['Time_day', 'Treatment', 'Force_percent']
TSS => ['Time_day', 'Treatment', 'TSS_percent']
O2 => ['Time_day', 'Treatment', 'percentO2']
CO2 => ['Time_day', 'Treatment', 'co2']


In [62]:
def standardize_sheet(sheet_name, df):
    df = df.copy()

    # Rename common columns
    rename_map = {
        "Time_day": "Day",
        "Treatment": "Treatment"
    }

    df = df.rename(columns=rename_map)

    # Specific rename by sheet
    if sheet_name == "morphology":
        df = df.rename(columns={
            "morphology_rate": "morphology"
        })

    elif sheet_name == "bacteria":
        df = df.rename(columns={
            "bacteria_total": "bacteria_total",
            "yeasts": "yeasts",
            "lactic": "lactic_bacteria"
        })

    elif sheet_name == "weight":
        df = df.rename(columns={
            "weight_loss_percent": "weight_loss"
        })

    elif sheet_name == "color":
        df = df.rename(columns={
            "delta_E": "color_delta_E"
        })

    elif sheet_name == "Texture analyzer":
        df = df.rename(columns={
            "Force_percent": "texture_force"
        })

    elif sheet_name == "TSS":
        df = df.rename(columns={
            "TSS_percent": "TSS"
        })

    elif sheet_name == "O2":
        df = df.rename(columns={
            "percentO2": "O2"
        })

    elif sheet_name == "CO2":
        df = df.rename(columns={
            "co2": "CO2"
        })

    # Keep only Day, Treatment and measurement columns
    keep_cols = ["Day", "Treatment"] + [
        col for col in df.columns
        if col not in ["Day", "Treatment"]
    ]

    df = df[keep_cols]

    return df


standardized_sheets = []

for sheet in expected_sheets:
    temp_df = standardize_sheet(sheet, clean_sheets[sheet])
    standardized_sheets.append(temp_df)

    print("\nPrepared sheet:", sheet)
    display(temp_df.head())


main_df = reduce(
    lambda left, right: pd.merge(left, right, on=["Day", "Treatment"], how="outer"),
    standardized_sheets
)

main_df = main_df.sort_values(["Treatment", "Day"]).reset_index(drop=True)

print("Main unified table shape:", main_df.shape)
display(main_df.head(20))


Prepared sheet: morphology


,Day,Treatment,morphology
0,0,control,5.00000
1,4,control,5.00000
2,7,control,4.00000
3,14,control,2.66667
4,0,CMC-3-methyl,5.00000



Prepared sheet: bacteria


,Day,Treatment,bacteria_total,yeasts,lactic_bacteria
0,0,control,6000.00,0,0.0
1,4,control,13888.89,0,220000.0
2,7,control,19805.56,0,385000.0
3,14,control,33611.11,0,770000.0
4,0,CMC-3-methyl,6555.56,0,0.0



Prepared sheet: weight


...

,Day,Treatment,O2
0,0,control,21.00000
1,4,control,20.61234
2,7,control,13.62778
3,14,control,0.00000
4,0,CMC-3-methyl,21.00000



Prepared sheet: CO2


,Day,Treatment,CO2
0,0,control,0.00000
1,4,control,0.34225
2,7,control,1.11594
3,14,control,2.92122
4,0,CMC-3-methyl,0.00000


Main unified table shape: (20, 12)


,Day,Treatment,morphology,bacteria_total,yeasts,lactic_bacteria,weight_loss,color_delta_E,texture_force,TSS,O2,CO2
0,0,CMC-3-methyl,5.00000,6555.56,0,0.00,0.00000,5.51961,8.26296,4.33333,21.00000,0.00000
1,4,CMC-3-methyl,5.00000,235000.00,0,28000.00,1.40953,11.44309,8.35616,4.46667,20.53906,0.30782
2,7,CMC-3-methyl,3.33333,406333.33,0,49000.00,2.33756,15.88570,7.43633,4.25000,14.50350,0.44798
3,14,CMC-3-methyl,3.00000,806111.11,0,98000.00,4.50297,26.25178,5.29007,3.74444,0.42052,0.77501
4,0,CMC-3-methyl+A,5.00000,34222.22,0,2422222.22,0.00000,5.35484,7.71785,4.66667,21.00000,0.00000
5,4,CMC-3-methyl+A,5.00000,2666666.67,0,344444.44,13.99081,9.74413,7.47227,5.46667,20.56355,0.29880
6,7,CMC-3-methyl+A,3.00000,4641000.00,0,0.00,3.68337,13.03609,6.66650,6.03333,15.27892,0.59220
7,14,CMC-3-methyl+A,3.33333,9247777.78,0,0.00,0.00000,20.71734,4.78636,7.35556,2.94814,1.27681
8,0,CMC-3-methyl+B,5.00000,6944.44,0,6155555.56,0.00000,3.59730,8.45524,6.10000,21.00000,0.00000
9,4,CMC-3-methyl+B,5.00000,221111.11,0,4122222.22,13.71022,11.58593,8.52326,6.50000,20.63471,0.12545


## Research Variables

### Dependent Variable

The main dependent variable is:

**Morphology**

Morphology represents the visible condition of the lettuce and is treated as the final ecological outcome of freshness or spoilage.

### Independent Variables

The independent variables are:

- Day
- Treatment
- CO2
- O2
- TSS
- Texture
- Weight loss
- Color change
- Bacteria total
- Yeasts
- Lactic bacteria

### Regression Formula

Morphology = β0  
+ β1·Day  
+ β2·Treatment  
+ β3·CO2  
+ β4·O2  
+ β5·TSS  
+ β6·Texture  
+ β7·WeightLoss  
+ β8·Color  
+ β9·Bacteria  
+ β10·Yeasts  
+ β11·LacticBacteria  
+ ε

This formula explains how the ecological, biological, chemical and physical criteria affect the visible quality of the product.

In [63]:
analysis_df = main_df.copy()

# Log transform for microbial variables because their scale is much larger
analysis_df["log_bacteria_total"] = np.log1p(analysis_df["bacteria_total"])
analysis_df["log_yeasts"] = np.log1p(analysis_df["yeasts"])
analysis_df["log_lactic_bacteria"] = np.log1p(analysis_df["lactic_bacteria"])

# Variables that usually indicate better freshness
positive_features = [
    "morphology",
    "texture_force",
    "O2",
    "log_lactic_bacteria"
]

# Variables that usually indicate degradation or spoilage
negative_features = [
    "CO2",
    "weight_loss",
    "color_delta_E",
    "log_bacteria_total",
    "log_yeasts"
]

# Some variables may not exist if names are different, so we filter safely
positive_features = [col for col in positive_features if col in analysis_df.columns]
negative_features = [col for col in negative_features if col in analysis_df.columns]

scaler = MinMaxScaler()

normalized_positive = pd.DataFrame(
    scaler.fit_transform(analysis_df[positive_features]),
    columns=positive_features
)

normalized_negative = pd.DataFrame(
    scaler.fit_transform(analysis_df[negative_features]),
    columns=negative_features
)

positive_score = normalized_positive.mean(axis=1)
negative_score = normalized_negative.mean(axis=1)

analysis_df["Quality_Score"] = 100 * (
    0.65 * positive_score +
    0.35 * (1 - negative_score)
)

analysis_df["Quality_Score"] = analysis_df["Quality_Score"].clip(0, 100)

display(analysis_df[[
    "Day",
    "Treatment",
    "morphology",
    "CO2",
    "O2",
    "weight_loss",
    "color_delta_E",
    "bacteria_total",
    "yeasts",
    "lactic_bacteria",
    "Quality_Score"
]].head(20))

,Day,Treatment,morphology,CO2,O2,weight_loss,color_delta_E,bacteria_total,yeasts,lactic_bacteria,Quality_Score
0,0,CMC-3-methyl,5.00000,0.00000,21.00000,0.00000,5.51961,6555.56,0,0.00,78.300540
1,4,CMC-3-methyl,5.00000,0.30782,20.53906,1.40953,11.44309,235000.00,0,28000.00,82.567572
2,7,CMC-3-methyl,3.33333,0.44798,14.50350,2.33756,15.88570,406333.33,0,49000.00,61.436133
3,14,CMC-3-methyl,3.00000,0.77501,0.42052,4.50297,26.25178,806111.11,0,98000.00,36.819917
4,0,CMC-3-methyl+A,5.00000,0.00000,21.00000,0.00000,5.35484,34222.22,0,2422222.22,90.264296
5,4,CMC-3-methyl+A,5.00000,0.29880,20.56355,13.99081,9.74413,2666666.67,0,344444.44,74.140911
6,7,CMC-3-methyl+A,3.00000,0.59220,15.27892,3.68337,13.03609,4641000.00,0,0.00,43.350311
7,14,CMC-3-methyl+A,3.33333,1.27681,2.94814,0.00000,20.71734,9247777.78,0,0.00,27.635802
8,0,CMC-3-methyl+B,5.00000,0.00000,21.00000,0.00000,3.59730,6944.44,0,6155555.56,95.605485
9,4,CMC-3-methyl+B,5.00000,0.12545,20.63471,13.71022,11.58593,221111.11,0,4122222.22,82.673632


In [64]:
fig = px.line(
    analysis_df,
    x="Day",
    y="Quality_Score",
    color="Treatment",
    markers=True,
    title="BioFreshGuard - Quality Score Over Time",
    labels={
        "Day": "Storage Day",
        "Quality_Score": "Quality Score",
        "Treatment": "Treatment"
    }
)

fig.update_layout(
    title_x=0.5,
    template="plotly_white",
    legend_title_text="Treatment"
)

fig.show()

In [65]:
treatment_summary = analysis_df.groupby("Treatment").agg(
    Mean_Quality=("Quality_Score", "mean"),
    Day_14_Quality=("Quality_Score", lambda x: x.iloc[-1]),
    Mean_Morphology=("morphology", "mean"),
    Mean_CO2=("CO2", "mean"),
    Mean_O2=("O2", "mean"),
    Mean_Bacteria=("bacteria_total", "mean")
).reset_index()

treatment_summary = treatment_summary.sort_values("Mean_Quality", ascending=False)

display(treatment_summary)

best_treatment = treatment_summary.iloc[0]["Treatment"]

print("Best treatment according to BioFreshGuard Quality Score:", best_treatment)

,Treatment,Mean_Quality,Day_14_Quality,Mean_Morphology,Mean_CO2,Mean_O2,Mean_Bacteria
3,CMC-3-methyl+C,80.732721,65.476095,4.250000,0.518563,15.851485,2.800260e+05
0,CMC-3-methyl,64.781040,36.819917,4.083333,0.382703,14.115770,3.635000e+05
2,CMC-3-methyl+B,63.731511,15.216615,4.166667,1.020852,15.158830,3.415799e+05
4,control,61.231809,36.626543,4.166667,1.094853,13.810030,1.832639e+04
1,CMC-3-methyl+A,58.847830,27.635802,4.083333,0.541952,14.947653,4.147417e+06


Best treatment according to BioFreshGuard Quality Score: CMC-3-methyl+C


## Initial Results Explanation

The unified BioFreshGuard database was built from the eight original experimental criteria only:
morphology, bacteria, weight, color, Texture analyzer, TSS, O2 and CO2.

The data were organized according to the main experimental days: 0, 4, 7 and 14.

A temporary Quality Score was calculated inside the notebook in order to summarize the overall freshness condition of each treatment over time. This score is not added as a new database criterion; it is used only as an analytical index for visualization, PCA, statistical modeling and dashboard development.

A higher Quality Score indicates better freshness preservation, while a lower score indicates stronger spoilage dynamics.

In [66]:
# Select numerical features for PCA
pca_features = [
    "Day",
    "morphology",
    "CO2",
    "O2",
    "TSS",
    "texture_force",
    "weight_loss",
    "color_delta_E",
    "log_bacteria_total",
    "log_yeasts",
    "log_lactic_bacteria",
    "Quality_Score"
]

# Keep only columns that exist
pca_features = [col for col in pca_features if col in analysis_df.columns]

X_pca = analysis_df[pca_features].copy()
X_pca = X_pca.fillna(X_pca.mean())

# Standardization is required before PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pca)

# PCA with two main components
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)

analysis_df["PC1"] = pca_result[:, 0]
analysis_df["PC2"] = pca_result[:, 1]

print("PCA Features:")
print(pca_features)

print("\nExplained Variance:")
print("PC1:", round(pca.explained_variance_ratio_[0], 3))
print("PC2:", round(pca.explained_variance_ratio_[1], 3))
print("Total explained variance:", round(pca.explained_variance_ratio_.sum(), 3))

PCA Features:
['Day', 'morphology', 'CO2', 'O2', 'TSS', 'texture_force', 'weight_loss', 'color_delta_E', 'log_bacteria_total', 'log_yeasts', 'log_lactic_bacteria', 'Quality_Score']

Explained Variance:
PC1: 0.552
PC2: 0.141
Total explained variance: 0.693


In [67]:
fig = px.scatter(
    analysis_df,
    x="PC1",
    y="PC2",
    color="Treatment",
    symbol="Day",
    size="Quality_Score",
    hover_data=["Day", "Treatment", "Quality_Score"],
    title="PCA Projection of BioFreshGuard Samples"
)

fig.update_layout(
    title_x=0.5,
    template="plotly_white"
)

fig.show()

In [68]:
loadings = pd.DataFrame(
    pca.components_.T,
    columns=["PC1", "PC2"],
    index=pca_features
)

display(loadings)

print("Strongest variables affecting PC1:")
display(loadings["PC1"].abs().sort_values(ascending=False).head(8))

print("Strongest variables affecting PC2:")
display(loadings["PC2"].abs().sort_values(ascending=False).head(8))

,PC1,PC2
Day,-0.384837,0.188573
morphology,0.359284,-0.062953
CO2,-0.347927,0.054681
O2,0.375774,-0.060266
TSS,0.010859,0.639601
texture_force,0.281048,0.428536
weight_loss,-0.126278,0.111285
color_delta_E,-0.356402,0.166497
log_bacteria_total,-0.249203,0.201138
log_yeasts,-0.000000,-0.000000


Strongest variables affecting PC1:


,PC1
Quality_Score,0.395209
Day,0.384837
O2,0.375774
morphology,0.359284
color_delta_E,0.356402
CO2,0.347927
texture_force,0.281048
log_bacteria_total,0.249203


Strongest variables affecting PC2:


,PC2
TSS,0.639601
log_lactic_bacteria,0.506694
texture_force,0.428536
log_bacteria_total,0.201138
Day,0.188573
color_delta_E,0.166497
Quality_Score,0.154343
weight_loss,0.111285


In [69]:
fig = px.scatter(
    analysis_df,
    x="PC1",
    y="PC2",
    color="Treatment",
    text="Day",
    size="Quality_Score",
    hover_data=["Treatment", "Day", "Quality_Score"],
    title="PCA Biplot - Samples and Criteria Directions"
)

scale_factor = 3

for feature in loadings.index:
    fig.add_shape(
        type="line",
        x0=0,
        y0=0,
        x1=loadings.loc[feature, "PC1"] * scale_factor,
        y1=loadings.loc[feature, "PC2"] * scale_factor,
        line=dict(width=1)
    )

    fig.add_annotation(
        x=loadings.loc[feature, "PC1"] * scale_factor,
        y=loadings.loc[feature, "PC2"] * scale_factor,
        text=feature,
        showarrow=False,
        font=dict(size=10)
    )

fig.update_layout(
    title_x=0.5,
    template="plotly_white"
)

fig.show()

## PCA Interpretation

Principal Component Analysis was used to reduce the dimensionality of the BioFreshGuard dataset and to identify the main ecological gradients that explain the differences between treatments and sampling days.

PC1 represents the strongest axis of variation in the dataset. If variables such as CO2, bacteria, yeasts, weight loss and color change have high loadings on PC1, this component can be interpreted as a spoilage or degradation axis.

PC2 represents the second strongest independent axis of variation. If morphology, O2, texture or lactic bacteria load strongly on PC2, this component may represent freshness preservation or biological protection.

The PCA biplot allows visual comparison between treatments and helps identify which criteria are most responsible for the separation between samples.

In [70]:
corr_features = [
    "Day",
    "morphology",
    "CO2",
    "O2",
    "TSS",
    "texture_force",
    "weight_loss",
    "color_delta_E",
    "log_bacteria_total",
    "log_yeasts",
    "log_lactic_bacteria",
    "Quality_Score",
    "PC1",
    "PC2"
]

corr_features = [col for col in corr_features if col in analysis_df.columns]

corr_matrix = analysis_df[corr_features].corr()

fig = px.imshow(
    corr_matrix,
    text_auto=True,
    aspect="auto",
    title="Correlation Matrix - BioFreshGuard Criteria"
)

fig.update_layout(
    title_x=0.5,
    template="plotly_white"
)

fig.show()

In [71]:
anova_model = ols("Quality_Score ~ C(Treatment)", data=analysis_df).fit()
anova_table = sm.stats.anova_lm(anova_model, typ=2)

display(anova_table)

p_value = anova_table["PR(>F)"].iloc[0]

print("ANOVA p-value:", p_value)

if p_value < 0.05:
    print("Result: Significant difference between treatments.")
else:
    print("Result: No statistically significant difference between treatments.")

,sum_sq,df,F,PR(>F)
C(Treatment),1189.931995,4.0,0.494973,0.73971
Residual,9015.126020,15.0,NaN,NaN


ANOVA p-value: 0.7397102896354854
Result: No statistically significant difference between treatments.


In [72]:
anova_morphology_model = ols("morphology ~ C(Treatment)", data=analysis_df).fit()
anova_morphology_table = sm.stats.anova_lm(anova_morphology_model, typ=2)

display(anova_morphology_table)

p_value_morphology = anova_morphology_table["PR(>F)"].iloc[0]

print("Morphology ANOVA p-value:", p_value_morphology)

if p_value_morphology < 0.05:
    print("Result: Significant difference between treatments in morphology.")
else:
    print("Result: No statistically significant difference between treatments in morphology.")

,sum_sq,df,F,PR(>F)
C(Treatment),0.077779,4.0,0.018454,0.999252
Residual,15.805554,15.0,NaN,NaN


Morphology ANOVA p-value: 0.9992517098201515
Result: No statistically significant difference between treatments in morphology.


## Statistical Hypothesis Testing

ANOVA was used to test whether the type of biological treatment has a statistically significant effect on lettuce quality.

### Null Hypothesis H0
There is no significant difference between treatments in product quality or morphology.

### Alternative Hypothesis H1
At least one treatment differs significantly from the others.

If the p-value is lower than 0.05, the null hypothesis is rejected and the treatment effect is considered statistically significant.

Two ANOVA tests were performed:
1. Treatment effect on Quality Score
2. Treatment effect on Morphology

In [73]:
regression_df = analysis_df.copy()

# Encode categorical Treatment column
regression_df = pd.get_dummies(regression_df, columns=["Treatment"], drop_first=True)

target = "morphology"

feature_candidates = [
    "Day",
    "CO2",
    "O2",
    "TSS",
    "texture_force",
    "weight_loss",
    "color_delta_E",
    "log_bacteria_total",
    "log_yeasts",
    "log_lactic_bacteria",
    "PC1",
    "PC2"
]

# Add encoded treatment columns
treatment_encoded_cols = [col for col in regression_df.columns if col.startswith("Treatment_")]
feature_candidates += treatment_encoded_cols

feature_cols = [col for col in feature_candidates if col in regression_df.columns]

X = regression_df[feature_cols].fillna(0)
y = regression_df[target]

linear_model = LinearRegression()
linear_model.fit(X, y)

y_pred = linear_model.predict(X)

print("Linear Regression Model - Predicting Morphology")
print("R2 score:", round(r2_score(y, y_pred), 3))
print("MAE:", round(mean_absolute_error(y, y_pred), 3))

coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": linear_model.coef_
}).sort_values("Coefficient", ascending=False)

display(coef_df)

fig = px.scatter(
    x=y,
    y=y_pred,
    labels={"x": "Actual Morphology", "y": "Predicted Morphology"},
    title="Actual vs Predicted Morphology - Linear Regression"
)

fig.add_shape(
    type="line",
    x0=y.min(),
    y0=y.min(),
    x1=y.max(),
    y1=y.max(),
    line=dict(width=2, dash="dash")
)

fig.update_layout(
    title_x=0.5,
    template="plotly_white"
)

fig.show()

Linear Regression Model - Predicting Morphology
R2 score: 1.0
MAE: 0.0


,Feature,Coefficient
10,PC1,1.898457e+00
1,CO2,8.639478e-01
7,log_bacteria_total,2.551572e-01
0,Day,1.480266e-01
6,color_delta_E,1.058450e-01
5,weight_loss,6.370420e-02
3,TSS,5.178973e-02
8,log_yeasts,8.326673e-17
15,Treatment_control,-1.129257e-15
12,Treatment_CMC-3-methyl+A,-1.534363e-15


In [74]:
rf_df = analysis_df.copy()
rf_df = pd.get_dummies(rf_df, columns=["Treatment"], drop_first=True)

target = "Quality_Score"

feature_candidates = [
    "Day",
    "morphology",
    "CO2",
    "O2",
    "TSS",
    "texture_force",
    "weight_loss",
    "color_delta_E",
    "log_bacteria_total",
    "log_yeasts",
    "log_lactic_bacteria",
    "PC1",
    "PC2"
]

treatment_encoded_cols = [col for col in rf_df.columns if col.startswith("Treatment_")]
feature_candidates += treatment_encoded_cols

feature_cols = [col for col in feature_candidates if col in rf_df.columns]

X = rf_df[feature_cols].fillna(0)
y = rf_df[target]

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=4,
    random_state=42
)

rf_model.fit(X, y)

rf_pred = rf_model.predict(X)

print("Random Forest Model - Predicting Quality Score")
print("R2 score:", round(r2_score(y, rf_pred), 3))
print("MAE:", round(mean_absolute_error(y, rf_pred), 3))

importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance_df)

fig = px.bar(
    importance_df,
    x="Importance",
    y="Feature",
    orientation="h",
    title="Random Forest Feature Importance"
)

fig.update_layout(
    title_x=0.5,
    template="plotly_white",
    yaxis=dict(autorange="reversed")
)

fig.show()

Random Forest Model - Predicting Quality Score
R2 score: 0.972
MAE: 2.855


,Feature,Importance
11,PC1,0.479080
3,O2,0.135022
1,morphology,0.090036
2,CO2,0.063910
5,texture_force,0.063661
7,color_delta_E,0.048282
0,Day,0.043438
8,log_bacteria_total,0.032765
10,log_lactic_bacteria,0.024003
4,TSS,0.005838


## Statistical and Predictive Modeling

Two modeling approaches were used:

### 1. Linear Regression
Linear regression was used to predict morphology as the main dependent variable.  
This model helps explain the direction and strength of the relationship between the criteria and the visible condition of the lettuce.

### 2. Random Forest Regression
Random Forest was used to predict the internal Quality Score.  
This model can capture non-linear relationships between biological, chemical and physical criteria.

The feature importance plot shows which variables contribute most to the prediction. This helps connect the model results back to the ecological interpretation of the system.

In [75]:
def create_initial_grid(size=12, initial_spoilage=0.08):
    grid = np.zeros((size, size))
    num_spoiled = int(size * size * initial_spoilage)

    spoiled_positions = np.random.choice(size * size, num_spoiled, replace=False)

    for pos in spoiled_positions:
        row = pos // size
        col = pos % size
        grid[row, col] = 1

    return grid


def count_spoiled_neighbors(grid, row, col):
    size = grid.shape[0]
    count = 0

    for dr in [-1, 0, 1]:
        for dc in [-1, 0, 1]:
            if dr == 0 and dc == 0:
                continue

            nr = row + dr
            nc = col + dc

            if 0 <= nr < size and 0 <= nc < size:
                count += grid[nr, nc]

    return count


def run_cellular_automata(
    size=12,
    steps=14,
    initial_spoilage=0.08,
    base_spoilage_probability=0.04,
    neighbor_effect=0.07,
    protection_effect=0.02
):
    grid = create_initial_grid(size=size, initial_spoilage=initial_spoilage)
    history = [grid.copy()]

    for step in range(steps):
        new_grid = grid.copy()

        for row in range(size):
            for col in range(size):

                if grid[row, col] == 1:
                    continue

                spoiled_neighbors = count_spoiled_neighbors(grid, row, col)

                probability = (
                    base_spoilage_probability
                    + neighbor_effect * spoiled_neighbors
                    - protection_effect
                )

                probability = max(0, min(1, probability))

                if np.random.rand() < probability:
                    new_grid[row, col] = 1

        grid = new_grid
        history.append(grid.copy())

    return history

In [76]:
np.random.seed(42)

scenarios = {
    "Baseline": {
        "initial_spoilage": 0.08,
        "base_spoilage_probability": 0.04,
        "neighbor_effect": 0.07,
        "protection_effect": 0.02
    },
    "Stress": {
        "initial_spoilage": 0.15,
        "base_spoilage_probability": 0.10,
        "neighbor_effect": 0.12,
        "protection_effect": 0.00
    },
    "Recovery": {
        "initial_spoilage": 0.06,
        "base_spoilage_probability": 0.03,
        "neighbor_effect": 0.05,
        "protection_effect": 0.04
    }
}

simulation_results = {}

for scenario_name, params in scenarios.items():
    history = run_cellular_automata(
        size=12,
        steps=14,
        initial_spoilage=params["initial_spoilage"],
        base_spoilage_probability=params["base_spoilage_probability"],
        neighbor_effect=params["neighbor_effect"],
        protection_effect=params["protection_effect"]
    )

    simulation_results[scenario_name] = history

    spoilage_percent = [grid.mean() * 100 for grid in history]

    fig = px.line(
        x=list(range(len(spoilage_percent))),
        y=spoilage_percent,
        markers=True,
        title=f"Spatial Spoilage Progression - {scenario_name}",
        labels={
            "x": "Simulation Step",
            "y": "Spoiled Area (%)"
        }
    )

    fig.update_layout(
        title_x=0.5,
        template="plotly_white"
    )

    fig.show()

In [77]:
for scenario_name, history in simulation_results.items():
    final_grid = history[-1]

    fig = px.imshow(
        final_grid,
        title=f"Final Spatial Spoilage Map - {scenario_name}",
        labels=dict(color="Spoilage State")
    )

    fig.update_layout(
        title_x=0.5,
        template="plotly_white"
    )

    fig.show()

In [78]:
scenario_summary = []

for scenario_name, history in simulation_results.items():
    final_spoilage = history[-1].mean() * 100
    initial_spoilage = history[0].mean() * 100

    scenario_summary.append({
        "Scenario": scenario_name,
        "Initial Spoilage (%)": initial_spoilage,
        "Final Spoilage (%)": final_spoilage,
        "Increase (%)": final_spoilage - initial_spoilage
    })

scenario_summary_df = pd.DataFrame(scenario_summary)
display(scenario_summary_df)

,Scenario,Initial Spoilage (%),Final Spoilage (%),Increase (%)
0,Baseline,7.638889,97.222222,89.583333
1,Stress,14.583333,100.000000,85.416667
2,Recovery,5.555556,60.416667,54.861111


## Spatial Simulation Model

A Cellular Automata model was implemented to represent spatial spoilage dynamics.

Each cell in the grid represents a lettuce storage unit or spatial sample area.

Cell states:
- 0 = Fresh
- 1 = Spoiled

The probability that a fresh cell becomes spoiled depends on:
1. Basic spoilage probability
2. Number of spoiled neighboring cells
3. Protection effect of biological treatment

Three scenarios were simulated:

### Baseline Scenario
Represents normal storage conditions.

### Stress Scenario
Represents disturbance conditions such as higher temperature, weaker protection, or faster microbial activity.

### Recovery Scenario
Represents improved biological protection and slower spoilage spread.

This model demonstrates spatial and temporal ecological dynamics and supports the interpretation of spoilage spread as a local-neighborhood process.

In [79]:
%%capture
!apt-get update -qq
!apt-get install -y ffmpeg
!pip -q install pykrige

In [80]:
# ============================================================
# Cell 35 - BioFreshGuard Final Dashboard
# Feature 1 + Feature 2
#
# Feature 1:
#   Graph 1  : Quality Score Over Time
#   Graph 1B : Morphology Over Time
#   Graph 2  : PCA Biplot
#   Graph 3  : Correlation Matrix
#
# Feature 2:
#   Selection: Treatment / Scenario / Microbe
#   Graph 0: Kriging + Game of Life + MP4 Spoilage Simulation
#   Buttons:
#       1. Selected Bacterial
#       2. CO2 and O2
#       3. Main Criteria
#       4. PCA Map
#       5. Scenario Comparison
# ============================================================

# ------------------------------------------------------------
# Colab widget support
# ------------------------------------------------------------
from google.colab import output as colab_output
colab_output.enable_custom_widget_manager()




# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------
import os
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
import plotly.express as px

from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import display, clear_output, HTML
from pykrige.ok import OrdinaryKriging

# ============================================================
# 0. Safety checks
# ============================================================

required_objects = ["analysis_df", "simulation_results", "scenarios"]

for obj in required_objects:
    if obj not in globals():
        raise NameError(f"Missing object: {obj}. Please run the previous cells before Cell 35.")

if "Quality_Score" not in analysis_df.columns:
    raise NameError("Missing column: Quality_Score. Please run the Quality Score cell before Cell 35.")

if "Day" not in analysis_df.columns:
    raise NameError("Missing column: Day.")

if "Treatment" not in analysis_df.columns:
    raise NameError("Missing column: Treatment.")

# ============================================================
# 1. Helper functions
# ============================================================

def file_to_base64(file_path):
    with open(file_path, "rb") as file:
        return base64.b64encode(file.read()).decode("utf-8")


def get_quality_status(value):
    if value >= 70:
        return "Fresh / Good"
    elif value >= 40:
        return "Warning"
    else:
        return "High Spoilage Risk"


def create_back_button():
    back_button = widgets.Button(
        description="Back to Home",
        button_style="warning",
        layout=widgets.Layout(width="200px", height="42px")
    )
    back_button.on_click(show_home)
    return back_button


def create_small_home_button():
    small_home_button = widgets.Button(
        description="Home",
        button_style="info",
        layout=widgets.Layout(width="120px", height="38px")
    )
    small_home_button.on_click(show_home)
    return small_home_button

# ============================================================
# 2. Create fresh vs rotten lettuce image for Home Page
# ============================================================

def create_fresh_rotten_lettuce_image(output_path="fresh_rotten_lettuce.png"):
    np.random.seed(42)

    size = 220
    y, x = np.ogrid[:size, :size]
    center = size / 2
    radius = size * 0.42
    mask = (x - center) ** 2 + (y - center) ** 2 <= radius ** 2

    def make_lettuce(rotten=False):
        img = np.ones((size, size, 3))
        img[:, :, :] = [0.96, 0.98, 0.95]

        lettuce = np.zeros((size, size, 3))

        if not rotten:
            lettuce[:, :, 0] = 0.25
            lettuce[:, :, 1] = 0.75
            lettuce[:, :, 2] = 0.25
        else:
            lettuce[:, :, 0] = 0.48
            lettuce[:, :, 1] = 0.42
            lettuce[:, :, 2] = 0.15

        texture = np.random.normal(0, 0.06, (size, size))
        lettuce[:, :, 1] += texture
        lettuce[:, :, 0] += texture * 0.5
        lettuce = np.clip(lettuce, 0, 1)

        img[mask] = lettuce[mask]

        if rotten:
            centers = [(70, 90), (135, 75), (115, 140), (85, 145), (150, 130)]
            for cx, cy in centers:
                spot = np.exp(-(((x - cx) ** 2 + (y - cy) ** 2) / (2 * 25 ** 2)))
                spot = spot * mask
                brown = np.array([0.25, 0.16, 0.06])
                for c in range(3):
                    img[:, :, c] = img[:, :, c] * (1 - spot * 0.75) + brown[c] * spot * 0.75

        return np.clip(img, 0, 1)

    fresh = make_lettuce(rotten=False)
    rotten = make_lettuce(rotten=True)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))

    axes[0].imshow(fresh)
    axes[0].set_title("Fresh Lettuce", fontsize=15)
    axes[0].axis("off")

    axes[1].imshow(rotten)
    axes[1].set_title("Rotten Lettuce", fontsize=15)
    axes[1].axis("off")

    plt.suptitle("Lettuce Freshness Comparison", fontsize=18, fontweight="bold")
    plt.tight_layout()
    plt.savefig(output_path, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close()

    return output_path


fresh_rotten_image_path = create_fresh_rotten_lettuce_image()
fresh_rotten_image_base64 = file_to_base64(fresh_rotten_image_path)

# ============================================================
# 3. Kriging + Game of Life MP4 function
# ============================================================

def create_kriging_game_of_life_mp4(
    temp_df,
    selected_treatment,
    selected_scenario,
    selected_variable,
    output_path="kriging_game_of_life_spoilage.mp4",
    grid_size=28,
    n_points=25,
    n_frames=22,
    random_seed=42
):
    """
    Creates an MP4 animation that combines:
    1. Ordinary Kriging spatial risk interpolation
    2. Game of Life / Cellular Automata spoilage spread

    Each grid cell represents a lettuce area.
    0 = Fresh
    1 = Spoiled
    """

    np.random.seed(random_seed)

    # --------------------------------------------------------
    # Scenario effect
    # --------------------------------------------------------

    scenario_name = str(selected_scenario).lower()

    if "stress" in scenario_name:
        scenario_factor = 1.45
        neighbor_factor = 0.20
        title_scenario = "Stress"
    elif "recovery" in scenario_name:
        scenario_factor = 0.65
        neighbor_factor = 0.08
        title_scenario = "Recovery"
    else:
        scenario_factor = 1.00
        neighbor_factor = 0.13
        title_scenario = "Baseline"

    # --------------------------------------------------------
    # Use selected treatment data to control spoilage level
    # --------------------------------------------------------

    final_quality = float(temp_df["Quality_Score"].iloc[-1])

    if selected_variable in temp_df.columns:
        microbe_value = float(temp_df[selected_variable].iloc[-1])
    else:
        microbe_value = 1.0

    microbe_log = np.log10(microbe_value + 1)
    microbe_effect = np.clip(microbe_log / 8, 0.05, 1.0)

    quality_risk = np.clip((100 - final_quality) / 100, 0.05, 0.95)

    base_spoilage_risk = np.clip(
        (0.35 * quality_risk + 0.65 * microbe_effect) * scenario_factor,
        0.05,
        0.95
    )

    # --------------------------------------------------------
    # Generate spatial sample points
    # --------------------------------------------------------

    x = np.random.rand(n_points) * 10
    y = np.random.rand(n_points) * 10

    def spoilage_risk_function(x, y):
        center_effect = np.exp(-((x - 5) ** 2 + (y - 5) ** 2) / 12)
        wave_effect = 0.20 * np.sin(x / 1.7) + 0.15 * np.cos(y / 2.0)
        return base_spoilage_risk + 0.35 * center_effect + wave_effect

    z = spoilage_risk_function(x, y) + np.random.normal(0, 0.05, n_points)
    z = np.clip(z, 0, 1)

    # --------------------------------------------------------
    # Ordinary Kriging interpolation
    # --------------------------------------------------------

    grid_x = np.linspace(0, 10, grid_size)
    grid_y = np.linspace(0, 10, grid_size)

    try:
        OK = OrdinaryKriging(
            x,
            y,
            z,
            variogram_model="spherical",
            verbose=False,
            enable_plotting=False
        )

        z_kriged, sigma = OK.execute("grid", grid_x, grid_y)
        risk_surface = np.array(z_kriged)

    except Exception as e:
        print("Kriging failed, using fallback spatial risk surface.")
        print(e)

        X, Y = np.meshgrid(grid_x, grid_y)
        risk_surface = spoilage_risk_function(X, Y)

    risk_surface = np.nan_to_num(risk_surface, nan=np.nanmean(risk_surface))

    risk_min = risk_surface.min()
    risk_max = risk_surface.max()

    if risk_max != risk_min:
        risk_surface = (risk_surface - risk_min) / (risk_max - risk_min)
    else:
        risk_surface = np.zeros_like(risk_surface)

    risk_surface = np.clip(risk_surface, 0, 1)

    # --------------------------------------------------------
    # Initial Game of Life grid
    # --------------------------------------------------------

    initial_probability = np.clip(0.03 + base_spoilage_risk * 0.12, 0.02, 0.20)
    grid = (np.random.rand(grid_size, grid_size) < initial_probability).astype(int)

    history = [grid.copy()]

    # --------------------------------------------------------
    # Cellular Automata update rule
    # --------------------------------------------------------

    for step in range(1, n_frames):
        new_grid = grid.copy()

        for i in range(grid_size):
            for j in range(grid_size):

                i_min = max(i - 1, 0)
                i_max = min(i + 2, grid_size)
                j_min = max(j - 1, 0)
                j_max = min(j + 2, grid_size)

                neighborhood = grid[i_min:i_max, j_min:j_max]
                spoiled_neighbors = neighborhood.sum() - grid[i, j]

                local_risk = risk_surface[i, j]

                if grid[i, j] == 0:
                    spoilage_probability = (
                        0.015 * scenario_factor
                        + neighbor_factor * spoiled_neighbors
                        + 0.12 * local_risk * scenario_factor
                    )

                    spoilage_probability = np.clip(spoilage_probability, 0, 0.95)

                    if np.random.rand() < spoilage_probability:
                        new_grid[i, j] = 1

                else:
                    if "recovery" in scenario_name:
                        if np.random.rand() < 0.015:
                            new_grid[i, j] = 0

        grid = new_grid.copy()
        history.append(grid.copy())

    # --------------------------------------------------------
    # Create MP4 animation
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(7, 6))

    img = ax.imshow(
        history[0],
        vmin=0,
        vmax=1,
        interpolation="nearest"
    )

    cbar = plt.colorbar(img, ax=ax)
    cbar.set_label("0 = Fresh, 1 = Spoiled")

    ax.set_xlabel("Grid X")
    ax.set_ylabel("Grid Y")

    text_box = ax.text(
        0.02,
        0.95,
        "",
        transform=ax.transAxes,
        fontsize=11,
        color="white",
        bbox=dict(boxstyle="round", facecolor="black", alpha=0.65)
    )

    def update(frame):
        current_grid = history[frame]
        img.set_data(current_grid)

        spoiled_percent = current_grid.mean() * 100

        ax.set_title(
            f"Kriging + Game of Life Spoilage Simulation\n"
            f"Treatment: {selected_treatment} | Scenario: {title_scenario}",
            fontsize=13
        )

        text_box.set_text(
            f"Step: {frame}\nSpoiled area: {spoiled_percent:.1f}%"
        )

        return img, text_box

    anim = FuncAnimation(
        fig,
        update,
        frames=len(history),
        interval=600,
        blit=False,
        repeat=True
    )

    try:
        writer = FFMpegWriter(fps=2, bitrate=1800)
        anim.save(output_path, writer=writer)
        plt.close(fig)
        return output_path

    except Exception as e:
        plt.close(fig)
        print("MP4 creation failed.")
        print(e)
        return None

# ============================================================
# 4. Global buttons
# ============================================================

home_button = widgets.Button(
    description="Home",
    button_style="info",
    layout=widgets.Layout(width="140px", height="45px")
)

feature1_button = widgets.Button(
    description="Feature 1: Main Ecological Visualization",
    button_style="success",
    layout=widgets.Layout(width="320px", height="36px")
)

feature2_button = widgets.Button(
    description="Feature 2: Treatment + Scenario Dashboard",
    button_style="primary",
    layout=widgets.Layout(width="340px", height="36px")
)

feature3_button = widgets.Button(
    description="Feature 3: Prediction + Recommendation",
    button_style="success",
    layout=widgets.Layout(width="360px", height="36px")
)

main_output = widgets.Output()

# ============================================================
# 5. Home Page
# ============================================================
def show_home(button=None):
    with main_output:
        clear_output(wait=True)

        # ====================================================
        # Main title
        # ====================================================
        display(HTML("""
        <div style="
            background: linear-gradient(90deg, #145A32, #2ECC71);
            padding: 30px;
            border-radius: 18px;
            color: white;
            font-family: Arial;
            margin-bottom: 24px;
            margin-top: 15px;
            box-shadow: 0 4px 14px rgba(0,0,0,0.15);
            text-align: center;">

            <h1 style="
                margin:0;
                font-size:44px;
                font-weight:700;">
                BioFreshGuard
            </h1>
        </div>
        """))

        # ====================================================
        # Buttons on the left, explanation on the right
        # ====================================================
        left_buttons = widgets.VBox(
            [feature1_button, feature2_button, feature3_button],
            layout=widgets.Layout(
                width="360px",
                gap="16px",
                align_items="stretch",
                justify_content="center"
            )
        )

        right_image = widgets.HTML("""
        <div style="
            background:#F8F9F9;
            padding:20px;
            border-radius:22px;
            text-align:center;
            font-family:Arial;
            box-shadow:0 3px 12px rgba(0,0,0,0.10);
            width:100%;">

            <h2 style="color:#145A32; margin-top:0;">
                Freshness vs Spoilage
            </h2>

            <p style="font-size:16px; max-width:720px; margin:auto;">
                The dashboard analyzes lettuce freshness and spoilage using biological,
                chemical, physical and spatial indicators.
            </p>
        </div>
        """)

        display(widgets.HBox(
            [left_buttons, right_image],
            layout=widgets.Layout(
                width="100%",
                align_items="center",
                justify_content="center",
                gap="24px",
                margin="0 0 24px 0"
            )
        ))
        # ====================================================
        # Explanation cards under the image/buttons
        # ====================================================
        display(HTML("""
        <div style="
            display:flex;
            gap:18px;
            flex-wrap:wrap;
            font-family:Arial;
            margin-bottom:22px;">

            <div style="
                flex:1;
                min-width:280px;
                background:#EAF7EE;
                padding:20px;
                border-radius:16px;
                border-left:7px solid #27AE60;
                box-shadow:0 2px 8px rgba(0,0,0,0.07);">
                <h3 style="margin-top:0;">Feature 1</h3>
                <p>
                Quality Score Over Time, Morphology Over Time, PCA Biplot and Correlation Matrix.
                </p>
            </div>

            <div style="
                flex:1;
                min-width:280px;
                background:#EAF2F8;
                padding:20px;
                border-radius:16px;
                border-left:7px solid #2980B9;
                box-shadow:0 2px 8px rgba(0,0,0,0.07);">
                <h3 style="margin-top:0;">Feature 2</h3>
                <p>
                Treatment and Scenario dashboard with Kriging + Game of Life MP4 simulation.
                </p>
            </div>

        </div>
        """))
        # ====================================================
        # Fresh vs Rotten image
        # ====================================================

        display(HTML(f"""
        <div style="
            background:#F8F9F9;
            padding:20px;
            border-radius:22px;
            text-align:center;
            font-family:Arial;
            margin-bottom:24px;
            box-shadow:0 3px 12px rgba(0,0,0,0.10);">

            <img src="data:image/png;base64,{fresh_rotten_image_base64}"
                 style="
                    max-width:570px;
                    width:68%;
                    border-radius:18px;
                    box-shadow:0 3px 12px rgba(0,0,0,0.18);
                 ">


        </div>
        """))

        # ====================================================
        # Buttons directly under the image
        # ====================================================

        display(widgets.HBox([
            feature1_button,
            feature2_button,
            feature3_button
        ], layout=widgets.Layout(
            justify_content="center",
            gap="22px",
            margin="0 0 24px 0"
        )))

        # ====================================================
        # Explanation cards under the buttons
        # ====================================================

        display(HTML("""
        <div style="
            display:flex;
            gap:18px;
            flex-wrap:wrap;
            font-family:Arial;
            margin-bottom:22px;">

            <div style="
                flex:1;
                min-width:280px;
                background:#EAF7EE;
                padding:20px;
                border-radius:16px;
                border-left:7px solid #27AE60;
                box-shadow:0 2px 8px rgba(0,0,0,0.07);">
                <h3 style="margin-top:0;">Feature 1</h3>
                <p>
                Quality Score Over Time, Morphology Over Time, PCA Biplot and Correlation Matrix.
                </p>
            </div>

            <div style="
                flex:1;
                min-width:280px;
                background:#EAF2F8;
                padding:20px;
                border-radius:16px;
                border-left:7px solid #2980B9;
                box-shadow:0 2px 8px rgba(0,0,0,0.07);">
                <h3 style="margin-top:0;">Feature 2</h3>
                <p>
                Treatment and Scenario dashboard with Kriging + Game of Life MP4 simulation.
                </p>
            </div>

        </div>
        """))

# ============================================================
# 6. Feature 1
# ============================================================

def show_feature1(button=None):
    with main_output:
        clear_output(wait=True)

        display(create_small_home_button())

        display(HTML("""
        <div style="
            background: linear-gradient(90deg, #145A32, #58D68D);
            padding: 26px;
            border-radius: 18px;
            color: white;
            font-family: Arial;
            margin-top:15px;
            margin-bottom:22px;">
            <h1 style="margin:0;">Feature 1</h1>
            <h2 style="margin:5px 0 0 0;">Main Ecological Visualization</h2>
            <p style="font-size:16px;">
            This feature presents the main visualization results:
            Quality Score over time, Morphology over time, PCA Biplot, and Correlation Matrix.
            </p>
        </div>
        """))

        # ====================================================
        # Graph output area on the right with scrolling
        # ====================================================

        feature1_graph_output = widgets.Output(
            layout=widgets.Layout(
                width="700px",
                max_height="820px",
                overflow_y="auto",
                overflow_x="auto",
                border="1px solid #E5E7E9",
                padding="12px"
            )
        )

        def show_feature1_menu():
            with feature1_graph_output:
                clear_output(wait=True)
                display(HTML("""
                <div style="
                    background:#FFFFFF;
                    padding:18px;
                    border-radius:14px;
                    border-left:6px solid #145A32;
                    font-family:Arial;
                    margin-bottom:15px;">
                    <h3>Feature 1 Graph Preview</h3>
                    <p>
                    Choose a graph from the buttons on the left. The selected graph will appear here.
                    </p>
                </div>
                """))

        def back_to_feature1_button():
            btn = widgets.Button(
                description="Back to Feature 1",
                button_style="warning",
                layout=widgets.Layout(width="180px", height="36px")
            )
            btn.on_click(lambda b: show_feature1_menu())
            return btn

        # ====================================================
        # Graph 1: Quality Score Over Time
        # ====================================================

        def show_quality_score_graph(button=None):
            with feature1_graph_output:
                clear_output(wait=True)

                display(back_to_feature1_button())

                display(HTML("""
                <div style="
                    background:#F8F9F9;
                    padding:14px;
                    border-radius:12px;
                    border-left:6px solid #27AE60;
                    font-family:Arial;
                    margin-bottom:15px;">
                    <h2>Graph 1: Quality Score Over Time</h2>
                    <p>
                    Shows how overall lettuce quality changes over storage days and compares treatment performance.
                    </p>
                </div>
                """))

                plt.figure(figsize=(10, 5))

                for treatment in sorted(analysis_df["Treatment"].unique()):
                    temp = analysis_df[analysis_df["Treatment"] == treatment].sort_values("Day").copy()

                    plt.plot(
                        temp["Day"],
                        temp["Quality_Score"],
                        marker="o",
                        linewidth=3,
                        markersize=8,
                        label=treatment
                    )

                plt.title("Quality Score Over Time by Treatment", fontsize=15)
                plt.xlabel("Storage Day", fontsize=12)
                plt.ylabel("Quality Score", fontsize=12)
                plt.ylim(0, 105)
                plt.grid(True, alpha=0.3)
                plt.legend(title="Treatment", bbox_to_anchor=(1.05, 1), loc="upper left")
                plt.tight_layout()
                plt.show()

        # ====================================================
        # Graph 1B: Morphology Over Time
        # ====================================================

        def show_morphology_graph(button=None):
            with feature1_graph_output:
                clear_output(wait=True)

                display(back_to_feature1_button())

                display(HTML("""
                <div style="
                    background:#F8F9F9;
                    padding:14px;
                    border-radius:12px;
                    border-left:6px solid #16A085;
                    font-family:Arial;
                    margin-bottom:15px;">
                    <h2>Graph 1B: Morphology Over Time</h2>
                    <p>
                    Tracks visual appearance deterioration as a direct indicator of spoilage progression.
                    </p>
                </div>
                """))

                if "morphology" in analysis_df.columns:
                    plt.figure(figsize=(10, 5))

                    for treatment in sorted(analysis_df["Treatment"].unique()):
                        temp = analysis_df[analysis_df["Treatment"] == treatment].sort_values("Day").copy()

                        plt.plot(
                            temp["Day"],
                            temp["morphology"],
                            marker="o",
                            linewidth=3,
                            markersize=8,
                            label=treatment
                        )

                    plt.title("Morphology Over Time by Treatment", fontsize=15)
                    plt.xlabel("Storage Day", fontsize=12)
                    plt.ylabel("Morphology / Visual Appearance", fontsize=12)
                    plt.grid(True, alpha=0.3)
                    plt.legend(title="Treatment", bbox_to_anchor=(1.05, 1), loc="upper left")
                    plt.tight_layout()
                    plt.show()
                else:
                    print("Morphology column was not found.")

        # ====================================================
        # Graph 2: PCA Biplot
        # ====================================================

        def show_pca_biplot_graph(button=None):
            with feature1_graph_output:
                clear_output(wait=True)

                display(back_to_feature1_button())

                display(HTML("""
                <div style="
                    background:#F8F9F9;
                    padding:14px;
                    border-radius:12px;
                    border-left:6px solid #2980B9;
                    font-family:Arial;
                    margin-bottom:15px;">
                    <h2>Graph 2: PCA Biplot</h2>
                    <p>
                    Summarizes physical, chemical and microbial variables and shows separation between samples.
                    </p>
                </div>
                """))

                if all(col in analysis_df.columns for col in ["PC1", "PC2"]) and "loadings" in globals():
                    plt.figure(figsize=(10, 6))

                    for treatment in sorted(analysis_df["Treatment"].unique()):
                        subset = analysis_df[analysis_df["Treatment"] == treatment]

                        plt.scatter(
                            subset["PC1"],
                            subset["PC2"],
                            s=90,
                            alpha=0.75,
                            label=treatment
                        )

                        if "Day" in subset.columns:
                            for _, row in subset.iterrows():
                                plt.text(
                                    row["PC1"],
                                    row["PC2"],
                                    str(row["Day"]),
                                    fontsize=8,
                                    alpha=0.75
                                )

                    scale_factor = 3

                    for feature in loadings.index:
                        plt.arrow(
                            0,
                            0,
                            loadings.loc[feature, "PC1"] * scale_factor,
                            loadings.loc[feature, "PC2"] * scale_factor,
                            head_width=0.05,
                            length_includes_head=True
                        )

                        plt.text(
                            loadings.loc[feature, "PC1"] * scale_factor,
                            loadings.loc[feature, "PC2"] * scale_factor,
                            feature,
                            fontsize=9
                        )

                    plt.axhline(0, linewidth=1)
                    plt.axvline(0, linewidth=1)
                    plt.title("PCA Biplot - PC1 vs PC2", fontsize=15)
                    plt.xlabel("PC1", fontsize=12)
                    plt.ylabel("PC2", fontsize=12)
                    plt.grid(True, alpha=0.3)
                    plt.legend(title="Treatment", bbox_to_anchor=(1.05, 1), loc="upper left")
                    plt.tight_layout()
                    plt.show()
                else:
                    print("PCA Biplot was not displayed. PC1, PC2 or loadings were not found.")

        # ====================================================
        # Graph 3: Correlation Matrix
        # ====================================================

        def show_correlation_matrix_graph(button=None):
            with feature1_graph_output:
                clear_output(wait=True)

                display(back_to_feature1_button())

                display(HTML("""
                <div style="
                    background:#F8F9F9;
                    padding:14px;
                    border-radius:12px;
                    border-left:6px solid #8E44AD;
                    font-family:Arial;
                    margin-bottom:15px;">
                    <h2>Graph 3: Correlation Matrix</h2>
                    <p>
                    Shows statistical relationships between quality, spoilage indicators, gases and microbial variables.
                    </p>
                </div>
                """))

                corr_features = [
                    "Day",
                    "morphology",
                    "CO2",
                    "O2",
                    "TSS",
                    "texture_force",
                    "weight_loss",
                    "color_delta_E",
                    "log_bacteria_total",
                    "log_yeasts",
                    "log_lactic_bacteria",
                    "Quality_Score",
                    "PC1",
                    "PC2"
                ]

                corr_features = [col for col in corr_features if col in analysis_df.columns]

                if len(corr_features) > 1:
                    corr_matrix = analysis_df[corr_features].corr()

                    fig = px.imshow(
                        corr_matrix,
                        text_auto=True,
                        aspect="auto",
                        title="Correlation Matrix - BioFreshGuard Criteria"
                    )

                    fig.update_layout(
                        title_x=0.5,
                        template="plotly_white"
                    )

                    fig.show()
                else:
                    print("Not enough numeric columns for Correlation Matrix.")

        # ====================================================
        # Buttons inside compact explanation blocks
        # ====================================================

        quality_button = widgets.Button(
            description="Quality Score Over Time",
            button_style="success",
            layout=widgets.Layout(width="100%", height="34px")
        )

        morphology_button = widgets.Button(
            description="Morphology Over Time",
            button_style="success",
            layout=widgets.Layout(width="100%", height="34px")
        )

        pca_button_f1 = widgets.Button(
            description="PCA Biplot",
            button_style="success",
            layout=widgets.Layout(width="100%", height="34px")
        )

        correlation_button = widgets.Button(
            description="Correlation Matrix",
            button_style="success",
            layout=widgets.Layout(width="100%", height="34px")
        )

        quality_button.on_click(show_quality_score_graph)
        morphology_button.on_click(show_morphology_graph)
        pca_button_f1.on_click(show_pca_biplot_graph)
        correlation_button.on_click(show_correlation_matrix_graph)

        def explanation_block(button_widget, explanation, border_color):
            return widgets.VBox([
                button_widget,
                widgets.HTML(f"""
                <div style="
                    background:#F8F9F9;
                    padding:6px 9px;
                    border-radius:0 0 9px 9px;
                    border-left:5px solid {border_color};
                    border-right:1px solid #E5E7E9;
                    border-bottom:1px solid #E5E7E9;
                    font-family:Arial;
                    width:100%;
                    box-sizing:border-box;
                    line-height:1.15;">
                    <span style="font-size:11px;">{explanation}</span>
                </div>
                """)
            ], layout=widgets.Layout(
                width="360px",
                margin="0 0 8px 0"
            ))

        left_menu = widgets.VBox([
            explanation_block(
                quality_button,
                "Shows overall lettuce quality across storage days and compares treatment performance.",
                "#27AE60"
            ),
            explanation_block(
                morphology_button,
                "Tracks visual appearance deterioration as a direct indicator of spoilage progression.",
                "#16A085"
            ),
            explanation_block(
                pca_button_f1,
                "Summarizes physical, chemical and microbial variables and shows separation between samples.",
                "#2980B9"
            ),
            explanation_block(
                correlation_button,
                "Shows statistical relationships between quality, spoilage indicators, gases and microbial variables.",
                "#8E44AD"
            )
        ], layout=widgets.Layout(
            width="380px"
        ))

        display(HTML("""
        <div style="
            background:#FFFFFF;
            padding:18px;
            border-radius:14px;
            border-left:6px solid #145A32;
            font-family:Arial;
            margin-bottom:15px;">
            <h3>Feature 1 Graph Navigation</h3>
            <p>
            Use the graph cards on the left. Each card contains a graph button and a short professional explanation.
            </p>
        </div>
        """))

        display(widgets.HBox([
            left_menu,
            feature1_graph_output
        ], layout=widgets.Layout(
            width="100%",
            align_items="flex-start",
            gap="20px",
            overflow="visible"
        )))

        show_feature1_menu()

        display(create_back_button())
# 7. Feature 2
# ============================================================

def show_feature2(button=None):
    with main_output:
        clear_output(wait=True)

        display(create_small_home_button())

        display(HTML("""
        <div style="
            background: linear-gradient(90deg, #1F618D, #5DADE2);
            padding: 28px;
            border-radius: 18px;
            color: white;
            font-family: Arial;
            margin-top:15px;
            margin-bottom:22px;">
            <h1 style="margin:0;">Feature 2</h1>
            <h2 style="margin:5px 0 0 0;">Treatment and Scenario Dashboard</h2>
            <p style="font-size:16px;">
            Select Treatment, Scenario and Microbial variable, then click Run Feature 2.
            </p>
        </div>
        """))

        treatment_dropdown_f2 = widgets.Dropdown(
            options=sorted(analysis_df["Treatment"].unique()),
            description="Treatment:",
            layout=widgets.Layout(width="520px")
        )

        scenario_dropdown_f2 = widgets.Dropdown(
            options=list(simulation_results.keys()),
            description="Scenario:",
            layout=widgets.Layout(width="520px")
        )

        microbial_options = [
            col for col in ["bacteria_total", "yeasts", "lactic_bacteria"]
            if col in analysis_df.columns
        ]

        if len(microbial_options) == 0:
            microbial_options = ["No microbial columns found"]

        variable_dropdown_f2 = widgets.Dropdown(
            options=microbial_options,
            description="Microbe:",
            layout=widgets.Layout(width="520px")
        )

        run_button = widgets.Button(
            description="Run Feature 2",
            button_style="primary",
            layout=widgets.Layout(width="260px", height="50px")
        )

        # Graph 0 output appears on the right side near the selection area
        feature2_output = widgets.Output(
            layout=widgets.Layout(
                width="520px",
                max_height="560px",
                overflow_y="auto",
                overflow_x="auto",
                border="1px solid #E5E7E9",
                padding="10px"
            )
        )

        # KPI cards and graph navigation appear below the selection area
        feature2_bottom_output = widgets.Output()

        selection_panel = widgets.VBox([
            widgets.HTML("""
            <div style="
                background:#F4F6F7;
                padding:18px;
                border-radius:14px;
                border-left:7px solid #1F618D;
                font-family:Arial;
                margin-bottom:18px;">
                <h3>Feature 2 Selection</h3>
                <p>Choose the treatment, scenario and microbial variable you want to analyze.</p>
            </div>
            """),
            treatment_dropdown_f2,
            scenario_dropdown_f2,
            variable_dropdown_f2,
            run_button
        ], layout=widgets.Layout(
            width="560px",
            gap="8px"
        ))

        display(widgets.HBox([
            selection_panel,
            feature2_output
        ], layout=widgets.Layout(
            width="100%",
            align_items="flex-start",
            gap="24px",
            margin="0 0 24px 0"
        )))

        display(feature2_bottom_output)

        def run_feature2(button=None):
            selected_treatment = treatment_dropdown_f2.value
            selected_scenario = scenario_dropdown_f2.value
            selected_variable = variable_dropdown_f2.value

            with feature2_output:
                clear_output(wait=True)

                display(HTML("""
                <div style="
                    background:#D5F5E3;
                    padding:14px;
                    border-radius:10px;
                    border-left:6px solid #27AE60;
                    font-family:Arial;
                    margin-top:10px;
                    margin-bottom:12px;">
                    <b>Run Feature 2 button works.</b><br>
                    Now generating a lighter Kriging + Game of Life MP4. Please wait a few seconds.
                </div>
                """))

            temp_df = analysis_df[
                analysis_df["Treatment"] == selected_treatment
            ].sort_values("Day").copy()

            if temp_df.empty:
                with feature2_output:
                    clear_output(wait=True)
                    print("No data found for selected treatment.")
                return

            start_quality = temp_df["Quality_Score"].iloc[0]
            final_quality = temp_df["Quality_Score"].iloc[-1]
            quality_change = final_quality - start_quality
            status = get_quality_status(final_quality)

            safe_treatment = str(selected_treatment).replace(" ", "_").replace("/", "_")
            safe_scenario = str(selected_scenario).replace(" ", "_").replace("/", "_")
            mp4_name = f"kriging_game_of_life_{safe_treatment}_{safe_scenario}.mp4"

            try:
                mp4_path = create_kriging_game_of_life_mp4(
                    temp_df=temp_df,
                    selected_treatment=selected_treatment,
                    selected_scenario=selected_scenario,
                    selected_variable=selected_variable,
                    output_path=mp4_name,
                    grid_size=18,
                    n_points=15,
                    n_frames=10,
                    random_seed=42
                )
            except Exception as e:
                mp4_path = None
                with feature2_output:
                    clear_output(wait=True)
                    display(HTML(f"""
                    <div style="
                        background:#FADBD8;
                        padding:14px;
                        border-radius:10px;
                        border-left:6px solid #C0392B;
                        font-family:Arial;
                        margin-top:18px;
                        margin-bottom:18px;">
                        <b>Error while generating MP4:</b><br>
                        {type(e).__name__}: {str(e)}
                    </div>
                    """))

            # ====================================================
            # Graph 0: Kriging + Game of Life MP4 on the right side
            # ====================================================

            with feature2_output:
                clear_output(wait=True)

                if mp4_path is not None and os.path.exists(mp4_path):
                    mp4_base64 = file_to_base64(mp4_path)

                    display(HTML(f"""
                    <div style="
                        background:#FFFFFF;
                        padding:12px;
                        border-radius:14px;
                        font-family:Arial;
                        box-shadow:0 2px 8px rgba(0,0,0,0.08);
                        margin-bottom:12px;">

                        <h3 style="margin-top:0;">Graph 0: Kriging + Game of Life Spoilage Simulation</h3>

                        <p style="font-size:13px;">
                        Kriging creates a spatial spoilage-risk surface, and Game of Life shows how spoilage
                        spreads between neighboring lettuce areas over time.
                        </p>

                        <p style="font-size:13px;">
                        <b>Treatment:</b> {selected_treatment}<br>
                        <b>Scenario:</b> {selected_scenario}<br>
                        <b>Microbial variable:</b> {selected_variable}
                        </p>

                        <video width="480" controls autoplay loop muted
                               style="
                                    max-width:100%;
                                    border-radius:12px;
                                    box-shadow:0 3px 12px rgba(0,0,0,0.18);
                               ">
                            <source src="data:video/mp4;base64,{mp4_base64}" type="video/mp4">
                            Your browser does not support the video tag.
                        </video>

                    </div>
                    """))

                else:
                    display(HTML("""
                    <div style="
                        background:#FADBD8;
                        padding:16px;
                        border-radius:12px;
                        border-left:6px solid #C0392B;
                        font-family:Arial;
                        margin-bottom:20px;">
                        <b>MP4 was not created.</b><br>
                        Please check that ffmpeg and pykrige are installed.
                    </div>
                    """))

            # ====================================================
            # KPI cards and graph navigation below the selection area
            # ====================================================

            with feature2_bottom_output:
                clear_output(wait=True)

                display(HTML(f"""
                <div style="display:flex; gap:15px; flex-wrap:wrap; font-family:Arial; margin-bottom:22px;">

                    <div style="flex:1; min-width:180px; background:#F4F6F7; padding:16px; border-radius:12px; border-left:6px solid #27AE60;">
                        <h4 style="margin:0;">Initial Quality</h4>
                        <h2 style="margin:8px 0;">{start_quality:.2f}</h2>
                    </div>

                    <div style="flex:1; min-width:180px; background:#F4F6F7; padding:16px; border-radius:12px; border-left:6px solid #2980B9;">
                        <h4 style="margin:0;">Final Quality</h4>
                        <h2 style="margin:8px 0;">{final_quality:.2f}</h2>
                    </div>

                    <div style="flex:1; min-width:180px; background:#F4F6F7; padding:16px; border-radius:12px; border-left:6px solid #F39C12;">
                        <h4 style="margin:0;">Quality Change</h4>
                        <h2 style="margin:8px 0;">{quality_change:.2f}</h2>
                    </div>

                    <div style="flex:1; min-width:180px; background:#F4F6F7; padding:16px; border-radius:12px; border-left:6px solid #8E44AD;">
                        <h4 style="margin:0;">Status</h4>
                        <h2 style="margin:8px 0;">{status}</h2>
                    </div>

                </div>
                """))

                feature2_graph_output = widgets.Output()

                def show_feature2_menu():
                    with feature2_graph_output:
                        clear_output(wait=True)
                        display(HTML(f"""
                        <div style="
                            background:#FFFFFF;
                            padding:18px;
                            border-radius:14px;
                            border-left:6px solid #1F618D;
                            font-family:Arial;
                            margin-bottom:15px;">
                            <h3>Feature 2 Graph Menu</h3>
                            <p><b>Treatment:</b> {selected_treatment}</p>
                            <p><b>Scenario:</b> {selected_scenario}</p>
                            <p><b>Selected microbial variable:</b> {selected_variable}</p>
                            <p>Choose a graph from the buttons on the left.</p>
                        </div>
                        """))

                def back_to_feature2_button():
                    btn = widgets.Button(
                        description="Back to Feature 2",
                        button_style="warning",
                        layout=widgets.Layout(width="220px", height="42px")
                    )
                    btn.on_click(lambda b: show_feature2_menu())
                    return btn

                def show_selected_bacterial(button=None):
                    with feature2_graph_output:
                        clear_output(wait=True)
                        display(back_to_feature2_button())

                        if selected_variable in temp_df.columns:
                            plt.figure(figsize=(11, 5))
                            plt.plot(
                                temp_df["Day"],
                                temp_df[selected_variable],
                                marker="o",
                                linewidth=3,
                                markersize=9
                            )
                            plt.title(
                                f"Selected Bacterial Variable Over Time\n{selected_variable} - {selected_treatment}",
                                fontsize=15
                            )
                            plt.xlabel("Storage Day")
                            plt.ylabel(selected_variable)
                            plt.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                        else:
                            print("Selected microbial variable was not found.")

                def show_gas_dynamics(button=None):
                    with feature2_graph_output:
                        clear_output(wait=True)
                        display(back_to_feature2_button())

                        if "CO2" in temp_df.columns and "O2" in temp_df.columns:
                            plt.figure(figsize=(11, 5))
                            plt.plot(temp_df["Day"], temp_df["CO2"], marker="o", linewidth=3, label="CO2")
                            plt.plot(temp_df["Day"], temp_df["O2"], marker="o", linewidth=3, label="O2")
                            plt.title(f"CO2 and O2 Gas Dynamics Over Time - {selected_treatment}", fontsize=15)
                            plt.xlabel("Storage Day")
                            plt.ylabel("Gas Level")
                            plt.legend()
                            plt.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                        else:
                            print("CO2 or O2 column was not found.")

                def show_main_criteria(button=None):
                    with feature2_graph_output:
                        clear_output(wait=True)
                        display(back_to_feature2_button())

                        criteria_cols = [
                            "morphology",
                            "weight_loss",
                            "color_delta_E",
                            "texture_force",
                            "TSS"
                        ]

                        criteria_cols = [col for col in criteria_cols if col in temp_df.columns]

                        if len(criteria_cols) > 0:
                            plt.figure(figsize=(11, 6))

                            for col in criteria_cols:
                                plt.plot(
                                    temp_df["Day"],
                                    temp_df[col],
                                    marker="o",
                                    linewidth=2.5,
                                    label=col
                                )

                            plt.title(f"Main Quality Criteria Over Time - {selected_treatment}", fontsize=15)
                            plt.xlabel("Storage Day")
                            plt.ylabel("Criterion Value")
                            plt.legend()
                            plt.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                        else:
                            print("No main criteria columns were found.")

                def show_pca_map(button=None):
                    with feature2_graph_output:
                        clear_output(wait=True)
                        display(back_to_feature2_button())

                        if "PC1" in analysis_df.columns and "PC2" in analysis_df.columns:
                            plt.figure(figsize=(9, 7))

                            for treatment in analysis_df["Treatment"].unique():
                                subset = analysis_df[
                                    analysis_df["Treatment"] == treatment
                                ]

                                plt.scatter(
                                    subset["PC1"],
                                    subset["PC2"],
                                    s=90,
                                    alpha=0.75,
                                    label=treatment
                                )

                            selected_points = analysis_df[
                                analysis_df["Treatment"] == selected_treatment
                            ]

                            plt.scatter(
                                selected_points["PC1"],
                                selected_points["PC2"],
                                s=230,
                                facecolors="none",
                                edgecolors="black",
                                linewidths=2.5,
                                label="Selected Treatment"
                            )

                            for _, row in selected_points.iterrows():
                                plt.text(
                                    row["PC1"],
                                    row["PC2"],
                                    str(row["Day"]),
                                    fontsize=10
                                )

                            plt.title("PCA Map - Selected Treatment Profile", fontsize=15)
                            plt.xlabel("PC1")
                            plt.ylabel("PC2")
                            plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
                            plt.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                        else:
                            print("PC1 or PC2 was not found.")

                def show_scenario_comparison(button=None):
                    with feature2_graph_output:
                        clear_output(wait=True)
                        display(back_to_feature2_button())

                        plt.figure(figsize=(11, 5))

                        for scenario_name, history in simulation_results.items():
                            spoilage_percent = [
                                grid.mean() * 100
                                for grid in history
                            ]

                            plt.plot(
                                range(len(spoilage_percent)),
                                spoilage_percent,
                                marker="o",
                                linewidth=2.5,
                                label=scenario_name
                            )

                        plt.title("Scenario Comparison - Spoiled Area Over Time", fontsize=15)
                        plt.xlabel("Simulation Step")
                        plt.ylabel("Spoiled Area (%)")
                        plt.legend()
                        plt.grid(True, alpha=0.3)
                        plt.tight_layout()
                        plt.show()

                bacterial_button = widgets.Button(
                    description="Selected Bacterial",
                    button_style="success",
                    layout=widgets.Layout(width="270px", height="44px")
                )

                gas_button = widgets.Button(
                    description="CO2 and O2",
                    button_style="success",
                    layout=widgets.Layout(width="270px", height="44px")
                )

                criteria_button = widgets.Button(
                    description="Main Criteria",
                    button_style="success",
                    layout=widgets.Layout(width="270px", height="44px")
                )

                pca_button = widgets.Button(
                    description="PCA Map",
                    button_style="success",
                    layout=widgets.Layout(width="270px", height="44px")
                )

                scenario_button = widgets.Button(
                    description="Scenario Comparison",
                    button_style="success",
                    layout=widgets.Layout(width="270px", height="44px")
                )

                bacterial_button.on_click(show_selected_bacterial)
                gas_button.on_click(show_gas_dynamics)
                criteria_button.on_click(show_main_criteria)
                pca_button.on_click(show_pca_map)
                scenario_button.on_click(show_scenario_comparison)

                side_buttons = widgets.VBox([
                    bacterial_button,
                    gas_button,
                    criteria_button,
                    pca_button,
                    scenario_button
                ], layout=widgets.Layout(
                    width="300px",
                    gap="10px",
                    border="1px solid #D5D8DC",
                    padding="12px"
                ))

                display(HTML("""
                <div style="
                    background:#FFFFFF;
                    padding:18px;
                    border-radius:14px;
                    border-left:6px solid #1F618D;
                    font-family:Arial;
                    margin-top:20px;
                    margin-bottom:15px;">
                    <h3>Feature 2 Graph Navigation</h3>
                    <p>
                    Use the buttons on the left to open each graph separately.
                    </p>
                </div>
                """))

                display(widgets.HBox([
                    side_buttons,
                    feature2_graph_output
                ], layout=widgets.Layout(
                    align_items="flex-start",
                    gap="20px"
                )))

                show_feature2_menu()

        run_button.on_click(run_feature2)

        display(create_back_button())





# ============================================================
# 8. Feature 3
# Future Prediction + Smart Recommendation
# ============================================================

def show_feature3(button=None):
    with main_output:
        clear_output(wait=True)

        display(create_small_home_button())

        display(HTML("""
        <div style="
            background: linear-gradient(90deg, #145A32, #58D68D);
            padding: 26px;
            border-radius: 18px;
            color: white;
            font-family: Arial;
            margin-top:15px;
            margin-bottom:22px;">
            <h1 style="margin:0;">Feature 3</h1>
            <h2 style="margin:5px 0 0 0;">Future Prediction and Smart Recommendation</h2>
            <p style="font-size:16px;">
            This feature predicts future Quality Score, evaluates fruit safety,
            and gives a smart eating/treatment recommendation.
            </p>
        </div>
        """))

        # ====================================================
        # Output area on the right
        # ====================================================

        feature3_graph_output = widgets.Output(
            layout=widgets.Layout(
                width="700px",
                max_height="820px",
                overflow_y="auto",
                overflow_x="auto",
                border="1px solid #E5E7E9",
                padding="12px"
            )
        )

        # ====================================================
        # Selection widgets
        # ====================================================

        treatment_dropdown_f3 = widgets.Dropdown(
            options=sorted(analysis_df["Treatment"].unique()),
            description="Treatment:",
            layout=widgets.Layout(width="360px")
        )

        if "simulation_results" in globals():
            scenario_options_f3 = list(simulation_results.keys())
        elif "scenarios" in globals():
            scenario_options_f3 = list(scenarios.keys())
        else:
            scenario_options_f3 = ["Baseline", "Stress", "Recovery"]

        scenario_dropdown_f3 = widgets.Dropdown(
            options=scenario_options_f3,
            description="Scenario:",
            layout=widgets.Layout(width="360px")
        )

        prediction_day_dropdown = widgets.Dropdown(
            options=[18, 21, 28],
            value=21,
            description="Day:",
            layout=widgets.Layout(width="360px")
        )

        # ====================================================
        # Helper functions
        # ====================================================

        def scenario_factor_value(scenario_name):
            scenario_name = str(scenario_name).lower()

            if "stress" in scenario_name:
                return 1.25
            elif "recovery" in scenario_name:
                return 0.80
            else:
                return 1.00

        def predict_series_by_treatment(df, value_col, future_days):
            df = df.dropna(subset=["Day", value_col]).sort_values("Day").copy()

            x = df["Day"].values.astype(float)
            y = df[value_col].values.astype(float)

            if len(x) < 2:
                return None

            degree = 2 if len(x) >= 3 else 1

            try:
                model = np.polyfit(x, y, degree)
                y_pred = np.polyval(model, future_days)
                return y_pred
            except Exception:
                return None

        def predict_quality_for_treatment(treatment_name, prediction_day, scenario_name):
            temp_df = analysis_df[
                analysis_df["Treatment"] == treatment_name
            ].sort_values("Day").copy()

            if temp_df.empty:
                return None

            pred = predict_series_by_treatment(
                temp_df,
                "Quality_Score",
                np.array([prediction_day], dtype=float)
            )

            if pred is None:
                return None

            predicted_quality = float(pred[0])
            factor = scenario_factor_value(scenario_name)

            if factor > 1:
                predicted_quality = predicted_quality - ((factor - 1) * 18)
            elif factor < 1:
                predicted_quality = predicted_quality + ((1 - factor) * 12)

            predicted_quality = float(np.clip(predicted_quality, 0, 100))
            return predicted_quality

        def get_prediction_status(predicted_quality):
            if predicted_quality >= 70:
                return "Low Risk / Good Quality"
            elif predicted_quality >= 40:
                return "Medium Risk / Warning"
            else:
                return "High Spoilage Risk"

        def safe_norm(value, column_name):
            if value is None or column_name not in analysis_df.columns:
                return 0

            max_value = analysis_df[column_name].max()

            if pd.isna(max_value) or max_value == 0:
                return 0

            return float(np.clip(value / max_value, 0, 1))

        def calculate_fruit_safety_index(selected_treatment, selected_scenario, prediction_day):
            temp_df = analysis_df[
                analysis_df["Treatment"] == selected_treatment
            ].sort_values("Day").copy()

            if temp_df.empty:
                return None

            future_days = np.array([0, 4, 7, 14, 18, 21, 28], dtype=float)
            day_index = np.where(future_days == prediction_day)[0][0]
            factor = scenario_factor_value(selected_scenario)

            quality_pred = predict_series_by_treatment(temp_df, "Quality_Score", future_days)

            if quality_pred is None:
                return None

            adjusted_quality_pred = quality_pred.copy()

            for idx, day in enumerate(future_days):
                if day > 14:
                    if factor > 1:
                        adjusted_quality_pred[idx] = adjusted_quality_pred[idx] - ((factor - 1) * 18)
                    elif factor < 1:
                        adjusted_quality_pred[idx] = adjusted_quality_pred[idx] + ((1 - factor) * 12)

            adjusted_quality_pred = np.clip(adjusted_quality_pred, 0, 100)
            predicted_quality = float(adjusted_quality_pred[day_index])

            predicted_morphology = None

            if "morphology" in temp_df.columns:
                morphology_pred = predict_series_by_treatment(temp_df, "morphology", future_days)
                if morphology_pred is not None:
                    predicted_morphology = float(morphology_pred[day_index])

            predicted_bacteria = None
            predicted_yeasts = None
            predicted_lactic = None

            if "bacteria_total" in temp_df.columns:
                bacteria_pred = predict_series_by_treatment(temp_df, "bacteria_total", future_days)
                if bacteria_pred is not None:
                    predicted_bacteria = float(np.clip(bacteria_pred[day_index] * factor, 0, None))

            if "yeasts" in temp_df.columns:
                yeasts_pred = predict_series_by_treatment(temp_df, "yeasts", future_days)
                if yeasts_pred is not None:
                    predicted_yeasts = float(np.clip(yeasts_pred[day_index] * factor, 0, None))

            if "lactic_bacteria" in temp_df.columns:
                lactic_pred = predict_series_by_treatment(temp_df, "lactic_bacteria", future_days)
                if lactic_pred is not None:
                    predicted_lactic = float(np.clip(lactic_pred[day_index], 0, None))

            bacteria_risk = safe_norm(predicted_bacteria, "bacteria_total")
            yeasts_risk = safe_norm(predicted_yeasts, "yeasts")
            lactic_protection = safe_norm(predicted_lactic, "lactic_bacteria")

            quality_component = predicted_quality / 100

            if predicted_morphology is not None and "morphology" in analysis_df.columns:
                morphology_max = analysis_df["morphology"].max()
                if pd.notna(morphology_max) and morphology_max != 0:
                    morphology_component = float(np.clip(predicted_morphology / morphology_max, 0, 1))
                else:
                    morphology_component = quality_component
            else:
                morphology_component = quality_component

            safety_index = 100 * (
                0.45 * quality_component +
                0.25 * morphology_component +
                0.15 * lactic_protection +
                0.10 * (1 - yeasts_risk) +
                0.05 * (1 - bacteria_risk)
            )

            safety_index = float(np.clip(safety_index, 0, 100))

            if safety_index >= 70:
                safety_status = "Recommended to Eat"
                safety_message = "The fruit/lettuce is predicted to remain in a good safety condition."
                status_color = "#27AE60"
            elif safety_index >= 45:
                safety_status = "Eat Soon / Check Before Eating"
                safety_message = "The fruit/lettuce may still be usable, but it should be checked carefully before eating."
                status_color = "#F39C12"
            else:
                safety_status = "Not Recommended to Eat"
                safety_message = "The fruit/lettuce is predicted to have high spoilage risk and is not recommended for eating."
                status_color = "#C0392B"

            components = {
                "Quality Score": quality_component * 100,
                "Morphology": morphology_component * 100,
                "Lactic Protection": lactic_protection * 100,
                "Yeasts Safety": (1 - yeasts_risk) * 100,
                "Total Bacteria Safety": (1 - bacteria_risk) * 100
            }

            return {
                "future_days": future_days,
                "quality_prediction": adjusted_quality_pred,
                "predicted_quality": predicted_quality,
                "predicted_morphology": predicted_morphology,
                "predicted_bacteria": predicted_bacteria,
                "predicted_yeasts": predicted_yeasts,
                "predicted_lactic": predicted_lactic,
                "safety_index": safety_index,
                "safety_status": safety_status,
                "safety_message": safety_message,
                "status_color": status_color,
                "components": components
            }

        def show_feature3_menu():
            with feature3_graph_output:
                clear_output(wait=True)
                display(HTML("""
                <div style="
                    background:#FFFFFF;
                    padding:18px;
                    border-radius:14px;
                    border-left:6px solid #145A32;
                    font-family:Arial;
                    margin-bottom:15px;">
                    <h3>Feature 3 Prediction Preview</h3>
                    <p>
                    Choose one of the prediction buttons on the left. The selected result will appear here.
                    </p>
                </div>
                """))

        def back_to_feature3_button():
            btn = widgets.Button(
                description="Back to Feature 3",
                button_style="warning",
                layout=widgets.Layout(width="180px", height="36px")
            )
            btn.on_click(lambda b: show_feature3_menu())
            return btn

        # ====================================================
        # Graph 1: Future Quality Prediction
        # ====================================================

        def show_quality_prediction_graph(button=None):
            with feature3_graph_output:
                clear_output(wait=True)

                display(back_to_feature3_button())

                selected_treatment = treatment_dropdown_f3.value
                selected_scenario = scenario_dropdown_f3.value
                prediction_day = prediction_day_dropdown.value

                temp_df = analysis_df[
                    analysis_df["Treatment"] == selected_treatment
                ].sort_values("Day").copy()

                future_days = np.array([0, 4, 7, 14, 18, 21, 28], dtype=float)

                quality_pred = predict_series_by_treatment(
                    temp_df,
                    "Quality_Score",
                    future_days
                )

                if quality_pred is None:
                    print("Not enough data to predict Quality Score.")
                    return

                factor = scenario_factor_value(selected_scenario)
                adjusted_quality_pred = quality_pred.copy()

                for idx, day in enumerate(future_days):
                    if day > 14:
                        if factor > 1:
                            adjusted_quality_pred[idx] = adjusted_quality_pred[idx] - ((factor - 1) * 18)
                        elif factor < 1:
                            adjusted_quality_pred[idx] = adjusted_quality_pred[idx] + ((1 - factor) * 12)

                adjusted_quality_pred = np.clip(adjusted_quality_pred, 0, 100)

                selected_day_quality = float(
                    adjusted_quality_pred[np.where(future_days == prediction_day)][0]
                )

                risk_status = get_prediction_status(selected_day_quality)

                display(HTML(f"""
                <div style="
                    background:#F8F9F9;
                    padding:14px;
                    border-radius:12px;
                    border-left:6px solid #27AE60;
                    font-family:Arial;
                    margin-bottom:15px;">
                    <h2>Graph 1: Predicted Quality Score</h2>
                    <p>
                    <b>Treatment:</b> {selected_treatment}<br>
                    <b>Scenario:</b> {selected_scenario}<br>
                    <b>Prediction day:</b> {prediction_day}<br>
                    <b>Predicted Quality:</b> {selected_day_quality:.2f}<br>
                    <b>Risk Level:</b> {risk_status}
                    </p>
                </div>
                """))

                observed_df = temp_df[["Day", "Quality_Score"]].dropna().sort_values("Day")

                plt.figure(figsize=(10, 5))

                plt.plot(
                    observed_df["Day"],
                    observed_df["Quality_Score"],
                    marker="o",
                    linewidth=3,
                    label="Observed Quality"
                )

                future_mask = future_days >= 14

                plt.plot(
                    future_days[future_mask],
                    adjusted_quality_pred[future_mask],
                    marker="o",
                    linestyle="--",
                    linewidth=3,
                    label="Predicted Quality"
                )

                plt.axvline(14, linestyle="--", alpha=0.6)
                plt.title(f"Future Quality Score Prediction - {selected_treatment}", fontsize=15)
                plt.xlabel("Storage Day")
                plt.ylabel("Quality Score")
                plt.ylim(0, 105)
                plt.grid(True, alpha=0.3)
                plt.legend()
                plt.tight_layout()
                plt.show()

        # ====================================================
        # Graph 2: Fruit Safety Index
        # ====================================================

        def show_fruit_safety_graph(button=None):
            with feature3_graph_output:
                clear_output(wait=True)

                display(back_to_feature3_button())

                selected_treatment = treatment_dropdown_f3.value
                selected_scenario = scenario_dropdown_f3.value
                prediction_day = prediction_day_dropdown.value

                result = calculate_fruit_safety_index(
                    selected_treatment,
                    selected_scenario,
                    prediction_day
                )

                if result is None:
                    print("Not enough data to calculate Fruit Safety Index.")
                    return

                display(HTML(f"""
                <div style="
                    background:#F8F9F9;
                    padding:14px;
                    border-radius:12px;
                    border-left:6px solid {result['status_color']};
                    font-family:Arial;
                    margin-bottom:15px;">
                    <h2>Graph 2: Fruit Safety Index</h2>
                    <p>
                    This graph evaluates whether the fruit/lettuce is recommended for eating.
                    The index combines predicted Quality Score, morphology, yeasts, total bacteria,
                    and lactic bacteria.
                    </p>
                    <p>
                    <b>Important biological interpretation:</b><br>
                    Lactic bacteria are not automatically treated as harmful.
                    They may represent beneficial biological activity that helps preserve the fruit.
                    </p>
                </div>
                """))

                display(HTML(f"""
                <div style="display:flex; gap:15px; flex-wrap:wrap; font-family:Arial; margin-bottom:22px;">

                    <div style="flex:1; min-width:190px; background:#F4F6F7; padding:16px; border-radius:12px; border-left:6px solid #27AE60;">
                        <h4 style="margin:0;">Fruit Safety Index</h4>
                        <h2 style="margin:8px 0;">{result['safety_index']:.2f}</h2>
                    </div>

                    <div style="flex:1; min-width:190px; background:#F4F6F7; padding:16px; border-radius:12px; border-left:6px solid {result['status_color']};">
                        <h4 style="margin:0;">Eating Recommendation</h4>
                        <h2 style="margin:8px 0;">{result['safety_status']}</h2>
                    </div>

                    <div style="flex:1; min-width:190px; background:#F4F6F7; padding:16px; border-radius:12px; border-left:6px solid #2980B9;">
                        <h4 style="margin:0;">Prediction Day</h4>
                        <h2 style="margin:8px 0;">Day {prediction_day}</h2>
                    </div>

                </div>
                """))

                plt.figure(figsize=(10, 5))
                plt.bar(result["components"].keys(), result["components"].values())
                plt.title(f"Fruit Safety Components - {selected_treatment}", fontsize=15)
                plt.ylabel("Component Score")
                plt.ylim(0, 105)
                plt.xticks(rotation=25, ha="right")
                plt.grid(axis="y", alpha=0.3)
                plt.tight_layout()
                plt.show()

                display(HTML(f"""
                <div style="
                    background:#EAF7EE;
                    padding:16px;
                    border-radius:12px;
                    border-left:6px solid {result['status_color']};
                    font-family:Arial;
                    margin-top:15px;">
                    <h3>Smart Eating Recommendation</h3>
                    <p><b>Status:</b> {result['safety_status']}</p>
                    <p>{result['safety_message']}</p>
                    <p>
                    The recommendation is based on predicted fruit quality, visual condition,
                    spoilage-related indicators, and the possible protective role of lactic bacteria.
                    </p>
                </div>
                """))

        # ====================================================
        # Graph 3: Smart Recommendation
        # ====================================================

        def show_recommendation_graph(button=None):
            with feature3_graph_output:
                clear_output(wait=True)

                display(back_to_feature3_button())

                selected_scenario = scenario_dropdown_f3.value
                prediction_day = prediction_day_dropdown.value

                recommendation_rows = []

                for treatment in sorted(analysis_df["Treatment"].unique()):
                    predicted_q = predict_quality_for_treatment(
                        treatment,
                        prediction_day,
                        selected_scenario
                    )

                    safety_result = calculate_fruit_safety_index(
                        treatment,
                        selected_scenario,
                        prediction_day
                    )

                    if predicted_q is not None and safety_result is not None:
                        recommendation_rows.append({
                            "Treatment": treatment,
                            "Predicted_Quality": predicted_q,
                            "Fruit_Safety_Index": safety_result["safety_index"]
                        })

                recommendation_df = pd.DataFrame(recommendation_rows)

                if recommendation_df.empty:
                    print("Could not create recommendation table.")
                    return

                recommendation_df = recommendation_df.sort_values(
                    ["Fruit_Safety_Index", "Predicted_Quality"],
                    ascending=False
                )

                best_treatment = recommendation_df.iloc[0]["Treatment"]
                best_quality = float(recommendation_df.iloc[0]["Predicted_Quality"])
                best_safety = float(recommendation_df.iloc[0]["Fruit_Safety_Index"])

                if best_safety >= 70:
                    best_status = "Recommended to Eat"
                elif best_safety >= 45:
                    best_status = "Eat Soon / Check Before Eating"
                else:
                    best_status = "Not Recommended to Eat"

                display(HTML(f"""
                <div style="
                    background:#F8F9F9;
                    padding:14px;
                    border-radius:12px;
                    border-left:6px solid #8E44AD;
                    font-family:Arial;
                    margin-bottom:15px;">
                    <h2>Graph 3: Smart Recommendation</h2>
                    <p>
                    <b>Best predicted treatment:</b> {best_treatment}<br>
                    <b>Predicted Quality Score:</b> {best_quality:.2f}<br>
                    <b>Fruit Safety Index:</b> {best_safety:.2f}<br>
                    <b>Eating Recommendation:</b> {best_status}
                    </p>
                    <p>
                    This recommendation is based on both predicted Quality Score and Fruit Safety Index.
                    Lactic bacteria are interpreted as possible protective microbes, not automatically as risk.
                    </p>
                </div>
                """))

                plt.figure(figsize=(10, 5))
                plt.bar(
                    recommendation_df["Treatment"],
                    recommendation_df["Fruit_Safety_Index"]
                )
                plt.title(f"Fruit Safety Index by Treatment - Day {prediction_day}", fontsize=15)
                plt.xlabel("Treatment")
                plt.ylabel("Fruit Safety Index")
                plt.ylim(0, 105)
                plt.xticks(rotation=30, ha="right")
                plt.grid(axis="y", alpha=0.3)
                plt.tight_layout()
                plt.show()

        # ====================================================
        # Buttons inside compact explanation blocks
        # ====================================================

        quality_prediction_button = widgets.Button(
            description="Quality Prediction",
            button_style="success",
            layout=widgets.Layout(width="100%", height="34px")
        )

        fruit_safety_button = widgets.Button(
            description="Fruit Safety Index",
            button_style="success",
            layout=widgets.Layout(width="100%", height="34px")
        )

        recommendation_button = widgets.Button(
            description="Smart Recommendation",
            button_style="success",
            layout=widgets.Layout(width="100%", height="34px")
        )

        # Explicit button-to-graph connections
        quality_prediction_button.on_click(show_quality_prediction_graph)
        fruit_safety_button.on_click(show_fruit_safety_graph)
        recommendation_button.on_click(show_recommendation_graph)

        def explanation_block(button_widget, explanation, border_color):
            return widgets.VBox([
                button_widget,
                widgets.HTML(f"""
                <div style="
                    background:#F8F9F9;
                    padding:6px 9px;
                    border-radius:0 0 9px 9px;
                    border-left:5px solid {border_color};
                    border-right:1px solid #E5E7E9;
                    border-bottom:1px solid #E5E7E9;
                    font-family:Arial;
                    width:100%;
                    box-sizing:border-box;
                    line-height:1.15;">
                    <span style="font-size:11px;">{explanation}</span>
                </div>
                """)
            ], layout=widgets.Layout(
                width="360px",
                margin="0 0 8px 0"
            ))

        left_menu = widgets.VBox([
            widgets.HTML("""
            <div style="
                background:#F4F6F7;
                padding:12px;
                border-radius:12px;
                border-left:6px solid #145A32;
                font-family:Arial;
                margin-bottom:10px;">
                <b>Feature 3 Selection</b>
            </div>
            """),
            treatment_dropdown_f3,
            scenario_dropdown_f3,
            prediction_day_dropdown,
            explanation_block(
                quality_prediction_button,
                "Predicts future Quality Score based on the observed trend.",
                "#27AE60"
            ),
            explanation_block(
                fruit_safety_button,
                "Evaluates if the fruit is recommended to eat using quality, morphology and microbial role.",
                "#1F618D"
            ),
            explanation_block(
                recommendation_button,
                "Compares treatments and recommends the best option using Quality and Fruit Safety Index.",
                "#8E44AD"
            )
        ], layout=widgets.Layout(
            width="380px",
            overflow="visible"
        ))

        display(HTML("""
        <div style="
            background:#FFFFFF;
            padding:18px;
            border-radius:14px;
            border-left:6px solid #145A32;
            font-family:Arial;
            margin-bottom:15px;">
            <h3>Feature 3 Graph Navigation</h3>
            <p>
            Use the selection fields and buttons on the left. The selected prediction or safety result will appear on the right.
            </p>
        </div>
        """))

        display(widgets.HBox([
            left_menu,
            feature3_graph_output
        ], layout=widgets.Layout(
            width="100%",
            align_items="flex-start",
            gap="20px",
            overflow="visible"
        )))

        show_feature3_menu()

        display(create_back_button())


# ============================================================
# 8. Display dashboard
# ============================================================

display(main_output)

home_button.on_click(show_home)
feature1_button.on_click(show_feature1)
feature2_button.on_click(show_feature2)
feature3_button.on_click(show_feature3)

show_home()



Output()